In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import os 
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# PLSSVD: fixed-dimension generalization to held-out trials

The number of components is fixed in advance (**5 by default**), controlled by `--n-components`. There is no tuning partition or component selection. Every split uses all available participants, with the same participant assignment and random-control source sampling throughout. Only trial partitions change. This evaluates **new trials from the same participants**, not unseen participants or unseen time points.

Default: **5 random 70/30 train/test splits**. These are repeated random holdouts, not five disjoint K-fold test sets. Trial sets can overlap between repetitions. `repeat=0` follows the same splitting procedure as every other split; it is special only for optional null diagnostics and example traces.

Run on the cluster with a new output directory:

```bash
python -u plssvd_eval.py --root /path/to/iEEGvsMEG \
  --meg-kind full_concatenated --n-components 5 --repeats 5 \
  --train-fraction 0.7 --output-dir /path/to/new_plssvd_eval_fixed
```

Copy the complete output directory locally, set `OUTPUT_DIR` below and run this reader notebook. Saved settings take precedence over the illustrative defaults below. Existing tuning-based outputs cannot supply the new metrics; run the updated script to obtain them.


In [ ]:
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
sys.path.insert(0, str(ROOT.parent / 'LB'))
from plssvd_eval_utils import (ValidationOptions, 
                               prepare_trial_cache, load_trial_cache,
                               validate_plssvd, plot_plssvd_validation, 
                               load_plssvd_results,)

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
MEG_KIND = 'full_concatenated'
# Illustrative batch defaults; this notebook loads results rather than running the analysis.
options = ValidationOptions(
    n_components=5, repeats=5, train_fraction=0.7,
    n_null=199, seed=2026, split_unit='trial', block_scaling='none',
)
CACHE_DIR = ROOT / 'out' / 'trial_cache'
OUTPUT_DIR = ROOT / 'out' / 'plssvd_eval_fixed'


## Load unaveraged trials
The original MEG `*_source.p` files contain trial averages and cannot support this validation. The exporter uses the `OUT.sources_ERFs` layout from `LB10_extract_data.py` and requires `mat73` when exporting MAT files. iEEG uses the existing epochs and event labels. Cache files are memory-mapped; export requires disk space for all selected trials. Actual MEG and iEEG time vectors must agree; an extra final MEG sample is removed only after checking its time vector.

Optional metadata CSV columns:

| Column | Meaning |
|---|---|
| modality | `ieeg` or `meg` |
| subject | Filename participant identifier |
| condition | Original condition code, e.g. 1 or 2 |
| trial_index | Zero-based trial index **within that condition** |
| split_group | Run or repeated stimulus identity; with `split_unit='group'`, never crosses partitions |
| permutation_block | Acoustic or other exchangeability stratum for condition-label shuffling |

When supplied, the CSV must cover every selected trial. Permutations also respect `split_group` when present. Use group splitting for dependent trials, including repeated stimuli where appropriate. Each condition needs at least six trials: at least two for training and two in each test half. The 70/30 fraction is adjusted for small counts. Group splitting requires at least three distinct groups and sufficient observations in both conditions; the fraction applies approximately to group counts. A cache made with different metadata/configuration must be rebuilt in a new directory.

All learned normalisation **in this notebook** is fitted on training data. This cannot undo leakage from upstream across-trial normalisation or data-derived source filters; those must be independent or fitted within folds for a fully strict evaluation.

In [ ]:
result = load_plssvd_results(OUTPUT_DIR)
if result['validation_options'].get('schema_version', 1) < 2:
    raise ValueError('These are legacy tuning-based results. Run the updated batch script with fixed --n-components into a new directory.')
MEG_KIND = result['validation_options']['meg_kind']
print(f"Loaded {MEG_KIND}: {result['validation_options']['repeats']} random train/test splits, "
      f"fixed k={result['validation_options']['n_components']}")
display(pd.Series(result['validation_options'], name='Saved evaluation settings'))
display(result['trial_counts'] if not result['trial_counts'].empty else result['participants'])


## Train once per split, then test — no tuning

Trials are randomly assigned within each participant, modality and condition to approximately 70% training and 30% testing. Test A and B divide that test set into halves solely for reliability; their union supplies the main test average. No trials are assigned to tuning. No neighboring time samples are randomly split. Both conditions are stacked along the observation axis.

MEG normalization, feature means, PLS projection weights and ridge cross-modal predictors use training averages only. Apply them unchanged to test averages. The same fixed k is used in every split; insufficient training rank causes an explicit error rather than silently choosing a smaller k. Test signs and component pairing are never optimized for the main performance metrics.

Each split starts from the same cached cohort and identical anatomical/participant assignment. Even split 0 holds out trials. Later splits refit preprocessing and all model parameters on a fresh random training set, without dropping participants. Predictions concern condition-averaged signals, not matched individual trials. Subject-level generalization is not assessed.


In [ ]:
display(result['summary'].round(3))
display(result['metric_summary'].query("partition == 'test'").round(3))
# Audit the actual proportions (small cohorts/groups may differ from 70/30).
display(result['split_audit'].groupby(['repeat', 'modality', 'partition']).size().unstack(fill_value=0))


## Goodness on held-out data: signal fit and shared covariance

**1. Own-modality reconstruction.** For each modality, project the test signals onto the training-learned PLS weight space and reconstruct them using the same weights:

\[
X_c=X_{test}-\mu_{train},\quad \widehat X_c=(X_cW)W^\top,\quad
R_{recon}=1-\frac{\|X_c-\widehat X_c\|_F^2}{\|X_c\|_F^2}.
\]

Any training whole-modality scale is undone for reconstruction. `ieeg_reconstruction_fraction` and `meg_reconstruction_fraction` report how much of each test signal the learned space retains. Test signals supply their own scores here, so this is reconstruction, not prediction from the other modality.

**2. Cross-modal prediction.** Training ridge regressions map MEG scores to iEEG features and iEEG scores to MEG features. `predict_ieeg_q2` and `predict_meg_q2` are `1 − test squared error / training-mean-baseline squared error`. Positive Q² improves on that baseline; negative Q² is worse. Training ridge settings and k are fixed in advance.

**3. Held-out cross-covariance.** Let C be the full test-feature cross-covariance after partition centering and the training-learned scaling, and let Wx/Wy be the training PLS weights. The test score covariance matrix is `Wx.T @ C @ Wy`. `crosscov_energy_fraction` is its squared Frobenius norm divided by `||C||F²`. The denominator is computed through sample Gram matrices, avoiding an enormous feature-by-feature C. This is the fraction of observed test cross-covariance energy retained by the trained spaces, not shared biological variance. Unlike training covariance, test score cross-covariance need not be diagonal.

Also report `mean_paired_covariance` (signed diagonal mean in native score units), per-component train/test covariance, `paired_crosscov_energy_fraction` (diagonal energy only), and signed paired score Pearson r. A covariance fraction can be high even when the absolute shared signal is weak, so inspect these measures together. Covariance retention relative to training is in `summary.csv` and can exceed one or become negative.

All curves compare train and test across random splits. There is no automatic model acceptance threshold. Goodness requires meaningful test performance in both modalities, retained covariance and reproducibility; none of these alone proves condition-specific biology.


In [ ]:
display(result['components'].round(3))
plot_plssvd_validation(result)


In [ ]:
# Descriptive consistency across independently refitted, overlapping splits.
display(result['fold_stability'].query("partition == 'test'").round(3))
display(result['metric_summary'].query("partition == 'test'").round(3))


## Stability across random splits

`metric_summary.csv` reports mean, SD, median and range of each train/test metric. The performance-consistency plot shows every test split. These ranges are descriptive, not population confidence intervals.

`fold_stability.csv` compares the refitted score time courses between every pair of splits, separately in iEEG and MEG. It uses mean absolute Pearson correlation with one-to-one component matching, handling sign flips and changes in order. `fold_component_pairs.csv` records the assignments. This post-hoc matching describes stability only and never changes test prediction or cross-covariance metrics. Components may rotate across nearly degenerate solutions; simple matching is not rotation invariant.

Test A/B correlations still quantify reliability conditional on one trained model. Across-split score correlations also include refitting variability and changes in trial averages, and test sets can overlap. They are not an independent spatial-weight stability analysis or a calibrated noise ceiling.

## Null models: distinct questions
Nulls are evaluated only for the primary fit, keeping preprocessing, fixed component count, and weights fixed. Statistics aggregate the selected components; no test-driven sign flips or component matching are applied.

1. **Temporal shift:** shifts all MEG test scores by the same circular offset within each condition. Tests sensitivity to temporal alignment while preserving circular temporal structure. Epoch-locked responses are nonstationary, so this tail fraction is a **surrogate diagnostic**, not automatically a valid permutation p-value.
2. **Condition labels:** permutes MEG held-out trial labels within participant and exchangeability blocks, preserving counts, then reconstructs projected condition contrasts using the same aggregation. A plus-one permutation tail is interpretable conditionally on exchangeable labels. Blocks containing only one condition cannot be shuffled. Acoustic confounding is not removed by simply labelling this a memory test.
3. **Channel correspondence:** permutes held-out MEG forward-pattern rows within iEEG participant and region (when available). Tests the specified spatial correspondence conditional on the fitted axes. Spatial autocorrelation can violate row exchangeability: treat this as a diagnostic unless those assumptions are justified.

Do not pool tail fractions across overlapping random splits. Report all planned tests and account for multiple inferential claims. The mean tone scaffold cancels in the condition difference only to the extent that it is shared across conditions. A recognition-memory claim still requires known condition semantics and adequate control of acoustic/sequence differences.

In [ ]:
display(result['null_tests'])
print(f'Results folder: {OUTPUT_DIR}')